# 04 — Context Managers
**References:** Ramalho *Fluent Python* Ch. 18 · Python docs: `contextlib` · PEP 343

## Narrative thread
```
with statement -> __enter__/__exit__ -> Exception handling -> contextlib -> Async context managers
```

## 1. The `with` statement and context manager protocol

The `with` statement guarantees that **teardown code always runs**, even if an exception is raised:

```python
with expr as var:
    body
```

Desugars to:
```python
mgr = expr
var = mgr.__enter__()
try:
    body
except:
    if not mgr.__exit__(*sys.exc_info()):
        raise       # re-raise if __exit__ returns falsy
else:
    mgr.__exit__(None, None, None)
```

**`__exit__` signature:** `(exc_type, exc_val, exc_tb)` — return `True` to suppress the exception, `False`/`None` to propagate it.

In [ ]:
import contextlib
import os
import sys
import tempfile
import threading
import time
import traceback
from pathlib import Path

# ── Class-based context manager ────────────────────────────────────────────
class ManagedFile:
    """Shows all __exit__ paths. Prefer built-in open() in production."""

    def __init__(self, path, mode='r'):
        self.path = path
        self.mode = mode
        self._file = None

    def __enter__(self):
        print(f'  __enter__: opening {self.path}')
        self._file = open(self.path, self.mode)
        return self._file           # value bound to 'as' target

    def __exit__(self, exc_type, exc_val, exc_tb):
        print(f'  __exit__:  closing {self.path}')
        if self._file:
            self._file.close()
        if exc_type is not None:
            print(f'  __exit__:  exception suppressed: {exc_type.__name__}: {exc_val}')
            return True             # True = suppress the exception
        return False                # False = don't suppress

# Normal exit
with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as tmp:
    tmp.write('hello world')
    tmppath = tmp.name

print('Normal exit:')
with ManagedFile(tmppath) as f:
    content = f.read()
print(f'  Read: {content!r}')

print()
print('Exit with exception (suppressed):')
with ManagedFile(tmppath) as f:
    raise ValueError('oops')
print('  continued after suppressed exception')
os.unlink(tmppath)

In [ ]:
# ── contextlib.contextmanager: generator-based CM ─────────────────────────
@contextlib.contextmanager
def timer(label: str = ''):
    """Measure elapsed time of the with block."""
    t0 = time.perf_counter()
    try:
        yield                       # point where the body runs
    finally:                        # always runs (exception or not)
        dt = time.perf_counter() - t0
        print(f'  [{label or "timer"}] {dt*1000:.3f} ms')

with timer('list-comprehension'):
    data = [x**2 for x in range(1_000_000)]

print()
# ── contextlib.suppress: swallow specific exceptions ──────────────────────
with contextlib.suppress(FileNotFoundError):
    os.unlink('/nonexistent/path')
print('FileNotFoundError suppressed cleanly')

print()
# ── contextlib.redirect_stdout/stderr ────────────────────────────────────
import io
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    print('This goes to the buffer, not the terminal')
print(f'Captured: {buf.getvalue()!r}')

print()
# ── contextlib.ExitStack: dynamic nesting ─────────────────────────────────
def process_many_files(paths: list[str]):
    """Open an unknown number of files — ExitStack handles cleanup."""
    with contextlib.ExitStack() as stack:
        files = []
        for p in paths:
            f = stack.enter_context(open(p))
            files.append(f)
        # All files closed when ExitStack exits, even if one fails mid-way
        return [f.name for f in files]

# Create some temp files
tmps = []
for i in range(3):
    tf = tempfile.NamedTemporaryFile(mode='w', suffix=f'_{i}.txt', delete=False)
    tf.write(f'file {i}')
    tf.close()
    tmps.append(tf.name)

names = process_many_files(tmps)
print(f'ExitStack opened {len(names)} files safely')
for p in tmps: os.unlink(p)

In [ ]:
# ── Reentrant vs non-reentrant context managers ───────────────────────────
# threading.Lock is NOT reentrant: same thread can deadlock itself
# threading.RLock IS reentrant: same thread can acquire multiple times

rlock = threading.RLock()

def recursive_work(n):
    with rlock:
        if n > 0:
            recursive_work(n - 1)
    return n

print(f'RLock reentrant (no deadlock): {recursive_work(3)}')

print()
# ── Async context manager ─────────────────────────────────────────────────
import asyncio

class AsyncTimer:
    def __init__(self, label=''):
        self.label = label

    async def __aenter__(self):
        self._t0 = time.perf_counter()
        return self

    async def __aexit__(self, exc_type, exc_val, exc_tb):
        dt = time.perf_counter() - self._t0
        print(f'  [async {self.label}] {dt*1000:.3f} ms')
        return False

async def demo_async_cm():
    async with AsyncTimer('sleep'):
        await asyncio.sleep(0.01)

asyncio.run(demo_async_cm())

# contextlib.asynccontextmanager for generator-style async CMs
@contextlib.asynccontextmanager
async def async_timer(label=''):
    t0 = time.perf_counter()
    try:
        yield
    finally:
        dt = time.perf_counter() - t0
        print(f'  [asynccontextmanager {label}] {dt*1000:.3f} ms')

async def demo2():
    async with async_timer('work'):
        await asyncio.sleep(0.005)

asyncio.run(demo2())

## 2. Key takeaways

| Concept | Rule |
|---|---|
| `__exit__` return | `True` suppresses the exception; `False`/`None` propagates it |
| `@contextmanager` | Generator-based: `yield` splits enter/exit; `finally` guarantees cleanup |
| `contextlib.suppress` | Concise exception swallowing with no try/except |
| `contextlib.ExitStack` | Dynamic number of CMs; rollback pattern |
| `RLock` vs `Lock` | `RLock` safe for reentrant code in same thread |
| `async with` | Requires `__aenter__`/`__aexit__` or `@asynccontextmanager` |

**Next:** notebook 05 — descriptors and metaclasses: the class creation machinery.